In [ ]:
# Fabric pipeline parameters
# Blank p_source_key = run all active Bronze jobs
p_source_key = "trips"
p_api_key = ""


In [ ]:
# Bronze ingestion: metadata-driven CSV + vehicle location API
# Reads active Bronze rows from admin_ops.admin_load_control
# Runs sequentially in one Spark session.
#
import time
import uuid
import hashlib
import traceback
from datetime import datetime, timezone

import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DoubleType, DecimalType, TimestampType
)

spark = SparkSession.builder.getOrCreate()

RUN_ID = str(uuid.uuid4())
STORAGE_ACCOUNT = "<storage_account>"
CONTROL_TABLE = "admin_ops.admin_load_control"
LOG_TABLE = "admin_ops.admin_processing_log"


VEHICLE_API_BASE_URL = "https://<vehicle-api-host>/data"
KEY_VAULT_URL = ""
API_KEY_SECRET_NAME = "vehicle-api-key"

# Write-retry tuning (applies to the Delta saveAsTable step).
WRITE_RETRY_ATTEMPTS = 3
WRITE_RETRY_BASE_DELAY_SEC = 5

# Partition sizing for writes: replaces the old hardcoded coalesce(1).
ROWS_PER_OUTPUT_PARTITION = 250_000
MIN_OUTPUT_PARTITIONS = 1
MAX_OUTPUT_PARTITIONS = 8


In [ ]:

# -------------------------------------------------------------------
# CSV schemas
# -------------------------------------------------------------------
TABLE_SCHEMAS = {
    "bronze_stop": StructType([
        StructField("STOP_ID", LongType()),
        StructField("STOP_ABBR", StringType()),
        StructField("STOP_NAME", StringType()),
        StructField("AT_STREET", StringType()),
        StructField("ON_STREET", StringType()),
        StructField("UNIT", StringType()),
        StructField("BAY", StringType()),
        StructField("CITY", StringType()),
        StructField("ZIPCODE", LongType()),
        StructField("ZONE", StringType()),
        StructField("STOP_LAT", DoubleType()),
        StructField("STOP_LNG", DoubleType()),
        StructField("STOP_TYPE", StringType()),
        StructField("PLACEMENT", StringType()),
        StructField("CORNER", StringType()),
        StructField("STATUS", StringType()),
        StructField("IS_WHEELCHAIR_ACCESSIBLE", IntegerType()),
        StructField("IS_BUS_PAD", IntegerType()),
        StructField("IS_LANDING_PAD", IntegerType()),
    ]),
    "bronze_schedule": StructType([
        StructField("SCHEDULE_ID", LongType()),
        StructField("SCHEDULE_DESC", StringType()),
        StructField("SCHEDULE_START_DATE", StringType()),
        StructField("SCHEDULE_END_DATE", StringType()),
        StructField("IS_DELETED", IntegerType()),
        StructField("IS_ACTIVE", IntegerType()),
    ]),
    "bronze_route": StructType([
        StructField("ROUTE_ID", LongType()),
        StructField("ROUTE_NAME", StringType()),
        StructField("ROUTE_TYPE", StringType()),
        StructField("DIVISION", StringType()),
        StructField("SERVICE_NAME", StringType()),
        StructField("SERVICE_DESCRIPTION", StringType()),
        StructField("ROUTE_GROUP", StringType()),
        StructField("ROUTE_GROUP_DESCRIPTION", StringType()),
        StructField("IS_HIGH_FREQUENCY_ROUTE", IntegerType()),
        StructField("IS_DELETED", IntegerType()),
    ]),
    "bronze_media_type": StructType([
        StructField("MEDIA_TYPE_ID", LongType()),
        StructField("MEDIA_TYPE_ABBR", StringType()),
        StructField("DESCRIPTION", StringType()),
        StructField("CATEGORY", StringType()),
    ]),
    "bronze_direction": StructType([
        StructField("DIRECTION_ID", LongType()),
        StructField("DIR_SHORT", StringType()),
        StructField("NUMBER_NOTATION", LongType()),
        StructField("DIRECTION_NAME", StringType()),
    ]),
    "bronze_block": StructType([
        StructField("BLOCK_ID", LongType()),
        StructField("SCHEDULE_ID", LongType()),
        StructField("BLOCK_NBR", StringType()),
        StructField("BLOCK_NAME", StringType()),
        StructField("DIVISION_ID", LongType()),
        StructField("STATUS", StringType()),
    ]),
    "bronze_trip": StructType([
        StructField("TRIP_ID", LongType()),
        StructField("SCHEDULE_ID", LongType()),
        StructField("TRIP_TYPE", StringType()),
        StructField("TRIP_START_TIME", StringType()),
        StructField("TRIP_END_TIME", StringType()),
        StructField("STATUS", StringType()),
    ]),
    "bronze_scheduled_trip": StructType([
        StructField("TRIP_ID", LongType()),
        StructField("SCHEDULE_ID", LongType()),
        StructField("ROUTE_ID", LongType()),
        StructField("STOP_ID", LongType()),
        StructField("STOP_SEQUENCE", LongType()),
        StructField("ROUTE_NAME", StringType()),
        StructField("STOP_NAME", StringType()),
        StructField("STOP_LAT", DoubleType()),
        StructField("STOP_LNG", DoubleType()),
        StructField("TRIP_TYPE", StringType()),
        StructField("TRIP_START_TIME", StringType()),
        StructField("TRIP_END_TIME", StringType()),
        StructField("TRIP_STATUS", StringType()),
    ]),
    "bronze_stop_activity": StructType([
        StructField("STOP_ID", LongType()),
        StructField("STOP_SEQ_ID", LongType()),
        StructField("SCHEDULE", LongType()),
        StructField("BLOCK", StringType()),
        StructField("MEDIA_TYPE", LongType()),
        StructField("VEHICLE", LongType()),
        StructField("RIDERSHIP_VALIDATION", LongType()),
        StructField("ROUTE", LongType()),
        StructField("TRIP_ID", LongType()),
        StructField("DIRECTION", LongType()),
        StructField("STOP_TIME", StringType()),
        StructField("PSGR_ON", LongType()),
        StructField("PSGR_OFF", LongType()),
        StructField("PSGR_LOAD", LongType()),
        StructField("TRIP_DATE", StringType()),
        StructField("LATITUDE", DoubleType()),
        StructField("LONGITUDE", DoubleType()),
        StructField("TRIP_MILES", DecimalType(38, 4)),
        StructField("PSGR_MILES", DecimalType(38, 4)),
        StructField("PSGR_HRS", DecimalType(38, 4)),
    ]),
}

DATE_TEXT_COLUMNS = {
    "bronze_unscheduled_trip": ["TRIP_DATE"],
    "bronze_schedule": ["SCHEDULE_START_DATE", "SCHEDULE_END_DATE"],
    "bronze_stop_activity": ["TRIP_DATE"],
}


In [ ]:
def source_path_exists(path):

    try:

        import notebookutils

        notebookutils.fs.ls(path)

        return True

    except Exception:

        try:

            from notebookutils import mssparkutils

            mssparkutils.fs.ls(path)

            return True

        except Exception:

            return False


def row_dict(row):
    return {str(k).lower(): v for k, v in row.asDict(recursive=True).items()}


def gv(fields, name, default=None):
    """fields is a pre-computed row_dict(row), not the raw Row.
    Avoids re-materializing the dict on every field lookup."""
    return fields.get(name.lower(), default)


def clean_str(value, default=""):
    if value is None:
        return default
    return str(value).strip()


def build_source_path(container, folder, file_name):
    container = clean_str(container)
    folder = clean_str(folder).strip("/")
    file_name = clean_str(file_name)

    if not container:
        raise ValueError("source_container is missing in control table.")
    if not file_name:
        raise ValueError("source_file_name is missing in control table.")

    if folder:
        return (
            f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
            f"{folder}/{file_name}"
        )

    return (
        f"abfss://{container}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
        f"{file_name}"
    )


def get_secret(vault_url, secret_name):
    # Fabric currently exposes notebookutils; older runtimes may expose mssparkutils.
    try:
        import notebookutils
        return notebookutils.credentials.getSecret(vault_url, secret_name)
    except Exception as first_error:
        try:
            from notebookutils import mssparkutils
            return mssparkutils.credentials.getSecret(vault_url, secret_name)
        except Exception:
            raise RuntimeError(
                f"Unable to retrieve Key Vault secret '{secret_name}'. "
                f"Check notebook identity/Key Vault access. Original error: {first_error}"
            )


def stable_row_id(*parts):
    """Deterministic ID derived from a hash of the parts. Replaces the old
    Python hash(), which is randomized per-process (PYTHONHASHSEED) and is
    not a reliable/reproducible dedup key across runs."""
    digest = hashlib.sha256("|".join(str(p) for p in parts).encode("utf-8")).hexdigest()
    return int(digest, 16) & 0x3FFFFFFFFFFFFFFF


def write_table_with_retry(writer, table_name, attempts=WRITE_RETRY_ATTEMPTS,
                            base_delay=WRITE_RETRY_BASE_DELAY_SEC):
    """Retries the actual Delta write with exponential backoff.

    This is the step that failed in job 188 (TASK_WRITE_FAILED writing to
    OneLake). Previously only the vehicle API calls had retry logic; the write
    itself — the thing that actually failed in production — had none."""
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            writer.saveAsTable(table_name)
            return
        except Exception as exc:
            last_error = exc
            if attempt == attempts:
                break
            delay = base_delay * attempt
            print(
                f"WARNING: write to {table_name} failed on attempt "
                f"{attempt}/{attempts}: {exc}. Retrying in {delay}s."
            )
            time.sleep(delay)

    raise last_error


def sized_partition_count(row_count):
    """Replaces the old hardcoded coalesce(1). Scales output partitions to
    the actual row count instead of forcing every table's writer through a
    single task, which was the likely cause of TASK_WRITE_FAILED: one
    oversized, non-parallel write to OneLake with no redundancy."""
    if row_count <= 0:
        return MIN_OUTPUT_PARTITIONS
    computed = (row_count // ROWS_PER_OUTPUT_PARTITION) + 1
    return max(MIN_OUTPUT_PARTITIONS, min(MAX_OUTPUT_PARTITIONS, computed))


def parse_key_columns(value):
    return [
        part.strip()
        for part in clean_str(value).split(",")
        if part.strip()
    ]


def get_key_columns(fields, destination_table):
    for column_name in (
        "primary_key_columns",
        "primary_key",
        "key_columns",
        "dedupe_key_columns",
    ):
        key_columns = parse_key_columns(gv(fields, column_name))
        if key_columns:
            return key_columns

    fallback_keys = {
        "bronze_stop_activity": [
            "STOP_ID",
            "STOP_SEQ_ID",
            "TRIP_ID",
            "TRIP_DATE",
        ],
    }

    return fallback_keys.get(destination_table, [])


def ensure_audit_columns(table_name):
    if not spark.catalog.tableExists(table_name):
        return

    existing = {c.upper() for c in spark.table(table_name).columns}
    additions = []
    if "INGESTED_AT" not in existing:
        additions.append("INGESTED_AT TIMESTAMP")
    if "UPDATED_AT" not in existing:
        additions.append("UPDATED_AT TIMESTAMP")

    if additions:
        spark.sql(
            f"ALTER TABLE {table_name} ADD COLUMNS ({', '.join(additions)})"
        )


def merge_incremental_bronze(stage_df, table_name, key_columns):
    if not key_columns:
        raise ValueError(
            f"{table_name}: key columns are required for INCREMENTAL load."
        )

    missing_keys = [c for c in key_columns if c not in stage_df.columns]
    if missing_keys:
        raise ValueError(
            f"{table_name}: key columns missing from source: {missing_keys}"
        )

    if not spark.catalog.tableExists(table_name):
        (
            stage_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        return

    ensure_audit_columns(table_name)

    target_columns = spark.table(table_name).columns
    for c in target_columns:
        if c not in stage_df.columns:
            stage_df = stage_df.withColumn(c, F.lit(None))

    view_name = f"__stg_{table_name.replace('.', '_')}_{RUN_ID[:8]}"
    stage_df.select(*target_columns).createOrReplaceTempView(view_name)

    on_expr = " AND ".join(
        [f"t.`{c}` <=> s.`{c}`" for c in key_columns]
    )

    update_parts = []
    for c in target_columns:
        if c in key_columns or c in {"INSERT_TS", "INGESTED_AT"}:
            continue
        if c in {"UPDATE_TS", "UPDATED_AT"}:
            update_parts.append(f"t.`{c}` = current_timestamp()")
        else:
            update_parts.append(f"t.`{c}` = s.`{c}`")

    insert_cols = ", ".join([f"`{c}`" for c in target_columns])
    insert_vals = ", ".join([f"s.`{c}`" for c in target_columns])

    spark.sql(f"""
        MERGE INTO {table_name} t
        USING `{view_name}` s
          ON {on_expr}
        WHEN MATCHED
          THEN UPDATE SET {", ".join(update_parts)}
        WHEN NOT MATCHED
          THEN INSERT ({insert_cols})
               VALUES ({insert_vals})
    """)

    spark.catalog.dropTempView(view_name)


def log_run(
    job_name,
    source_name,
    destination_table,
    status,
    total_rows,
    start_ts,
    information=None,
    error_message=None
):
    end_ts = datetime.now(timezone.utc)
    row_id = stable_row_id(RUN_ID, job_name, source_name, end_ts.isoformat())

    log_df = spark.createDataFrame(
        [(
            row_id,
            RUN_ID,
            job_name,
            start_ts,
            end_ts,
            int((end_ts - start_ts).total_seconds()),
            status,
            information,
            int(total_rows),
            None if status == "SUCCESS" else str(uuid.uuid4()),
            error_message,
            source_name,
            destination_table,
            None,
            None,
            None,
            None,
            end_ts
        )],
        schema=StructType([
            StructField("ROW_ID", LongType()),
            StructField("EXECUTION_ID", StringType()),
            StructField("JOB_NAME", StringType()),
            StructField("JOB_START_TS", TimestampType()),
            StructField("JOB_END_TS", TimestampType()),
            StructField("DURATION_SEC", LongType()),
            StructField("JOB_STATUS", StringType()),
            StructField("INFORMATION", StringType()),
            StructField("TOTAL_ROWS", LongType()),
            StructField("ERROR_ID", StringType()),
            StructField("ERROR_MESSAGE", StringType()),
            StructField("SOURCE_TABLE", StringType()),
            StructField("DESTINATION_TABLE", StringType()),
            StructField("PULL_FROM_DATE", TimestampType()),
            StructField("PULL_TO_DATE", TimestampType()),
            StructField("PROCESSED_CONDITION", StringType()),
            StructField("FLAG", StringType()),
            StructField("INSERT_TS", TimestampType()),
        ])
    )

    try:
        log_df.write.mode("append").saveAsTable(LOG_TABLE)
    except Exception as log_error:
        print(f"WARNING: failed to write log for {job_name}: {log_error}")


def is_api_row(fields):
    source_type = clean_str(gv(fields, "source_type")).upper()
    ingestion_method = clean_str(gv(fields, "ingestion_method")).upper()
    source_name = clean_str(gv(fields, "source_name")).lower()

    return (
        source_type == "API"
        or "API" in ingestion_method
        or "REST" in ingestion_method
        or source_name == "vehicle"
    )



# -------------------------------------------------------------------
# Fabric filesystem helper
# -------------------------------------------------------------------
def get_fs():
    try:
        import notebookutils
        return notebookutils.fs
    except Exception:
        from notebookutils import mssparkutils
        return mssparkutils.fs


# -------------------------------------------------------------------
# Archive file only after Bronze write succeeds, then delete source.
# -------------------------------------------------------------------
def archive_and_delete_source(
    source_container,
    source_folder,
    source_file_name,
    archive_container,
    archive_folder,
):
    source_container = clean_str(source_container)
    source_folder = clean_str(source_folder).strip("/")
    source_file_name = clean_str(source_file_name)

    archive_container = clean_str(archive_container)
    archive_folder = clean_str(archive_folder).strip("/")

    if not archive_container:
        raise ValueError(
            f"{source_file_name}: ARCHIVE_CONTAINER is missing in {CONTROL_TABLE}."
        )

    if not archive_folder:
        raise ValueError(
            f"{source_file_name}: ARCHIVE_FOLDER is missing in {CONTROL_TABLE}."
        )

    source_path = build_source_path(
        source_container,
        source_folder,
        source_file_name,
    )

    now_utc = datetime.now(timezone.utc)
    date_folder = now_utc.strftime("%Y/%m/%d")
    run_folder = f"{now_utc.strftime('%H%M%S')}_{RUN_ID[:8]}"

    archive_run_folder = (
        f"{archive_folder}/{date_folder}/{run_folder}"
    )

    archive_path = build_source_path(
        archive_container,
        archive_run_folder,
        source_file_name,
    )

    archive_parent = archive_path.rsplit("/", 1)[0]
    fs = get_fs()

    print("-" * 80)
    print(f"ARCHIVE SOURCE: {source_path}")
    print(f"ARCHIVE TARGET: {archive_path}")

    # Create archive directory.
    fs.mkdirs(archive_parent)

    # Copy first. If copy fails, delete is never attempted.
    fs.cp(
        source_path,
        archive_path,
        False,
    )

    print("ARCHIVE COPY  : SUCCESS")

    # Delete original only after copy succeeds.
    fs.rm(
        source_path,
        False,
    )

    print("SOURCE DELETE : SUCCESS")
    print("-" * 80)

    return archive_path


# -------------------------------------------------------------------
# Excel / XLSX helpers
# -------------------------------------------------------------------
def read_excel_source(source_path, source_name, source_file_name):
    import os
    import pandas as pd

    safe_file_name = source_file_name.replace("/", "_").replace("\\\\", "_")
    local_path = f"/tmp/{RUN_ID[:8]}_{safe_file_name}"
    local_uri = f"file:{local_path}"

    try:
        import notebookutils
        notebookutils.fs.cp(source_path, local_uri, True)
    except Exception:
        from notebookutils import mssparkutils
        mssparkutils.fs.cp(source_path, local_uri, True)

    try:
        sheet_name = "Unscheduled Trips" if source_name.lower() == "unscheduled_trip" else 0

        pdf = pd.read_excel(
            local_path,
            sheet_name=sheet_name,
            dtype=str,
        )

        pdf.columns = [
            str(c).strip().upper().replace(" ", "_")
            for c in pdf.columns
        ]

        # Convert pandas NaN/NaT to Python None before Spark conversion.
        pdf = pdf.where(pd.notnull(pdf), None)

        return spark.createDataFrame(pdf)

    finally:
        try:
            os.remove(local_path)
        except Exception:
            pass


def normalize_unscheduled_trip(df):
    expected_columns = [
        "TRIP_DATE",
        "TRIP_TYPE",
        "TRIP_DESC",
        "BLOCK_ID",
        "VEHICLE_ID",
        "ROUTE_ID",
        "DIRECTION_ID",
        "START_TIME",
        "END_TIME",
        "NUMBER_OF_VEHICLES",
        "RIDERSHIP",
    ]

    for c in expected_columns:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast(StringType()))

    # Keep Bronze close to source, but normalize the fields that Gold consumes.
    df = (
        df
        .withColumn(
            "TRIP_DATE",
            F.coalesce(
                F.to_date(F.col("TRIP_DATE"), "dd-MM-yyyy"),
                F.to_date(F.col("TRIP_DATE"), "MM/dd/yyyy"),
                F.to_date(F.col("TRIP_DATE"), "yyyy-MM-dd"),
                F.to_date(F.col("TRIP_DATE")),
            ),
        )
        .withColumn("BLOCK_ID", F.col("BLOCK_ID").cast(LongType()))
        .withColumn("VEHICLE_ID", F.col("VEHICLE_ID").cast(LongType()))
        .withColumn("ROUTE_ID", F.col("ROUTE_ID").cast(LongType()))
        .withColumn("DIRECTION_ID", F.col("DIRECTION_ID").cast(LongType()))
        .withColumn("NUMBER_OF_VEHICLES", F.col("NUMBER_OF_VEHICLES").cast(LongType()))
        .withColumn("RIDERSHIP", F.col("RIDERSHIP").cast(LongType()))
        .select(*expected_columns)
    )

    return df


# -------------------------------------------------------------------
# CSV ingestion
# -------------------------------------------------------------------
def ingest_csv_row(row):

    fields = row_dict(row)

    source_name = clean_str(
        gv(fields, "source_name")
    )

    source_container = clean_str(
        gv(fields, "source_container")
    )

    source_folder = clean_str(
        gv(fields, "source_folder")
    )

    source_file_name = clean_str(
        gv(fields, "source_file_name")
    )

    archive_container = clean_str(
        gv(fields, "archive_container")
    )

    archive_folder = clean_str(
        gv(fields, "archive_folder")
    )

    destination_schema = clean_str(
        gv(
            fields,
            "destination_schema"
        ),
        "bronze"
    ).lower()

    destination_table = clean_str(
        gv(
            fields,
            "destination_table"
        )
    ).lower()

    load_type = clean_str(
        gv(
            fields,
            "load_type"
        ),
        "FULL"
    ).upper()

    incremental_column = clean_str(
        gv(
            fields,
            "incremental_column"
        )
    )

    key_columns = get_key_columns(
        fields,
        destination_table
    )

    job_name = clean_str(
        gv(
            fields,
            "job_name"
        ),
        destination_table
    )

    start_ts = datetime.now(
        timezone.utc
    )

    out_df = None

    try:

        # ===========================================================
        # BUILD SOURCE PATH
        # ===========================================================

        source_path = build_source_path(
            source_container,
            source_folder,
            source_file_name,
        )

        print("=" * 80)
        print(f"FILE JOB     : {job_name}")
        print(f"SOURCE       : {source_path}")
        print(
            f"DESTINATION  : "
            f"{destination_schema}."
            f"{destination_table}"
        )
        print(f"LOAD TYPE    : {load_type}")


        # ===========================================================
        # CHECK SOURCE FILE EXISTS
        #
        # Missing inbound file is always treated as WARNING / SKIP.
        # This applies whether p_source_key is blank or a specific
        # source was requested. This allows pipeline test runs to
        # complete even when a source file has already been archived
        # and deleted from the inbound location.
        # ===========================================================

        if not source_path_exists(
            source_path
        ):

            message = (
                "Source file does not exist: "
                f"{source_path}"
            )

            print(
                f"WARNING: {message}"
            )

            print(
                f"SKIPPING     : "
                f"{job_name}"
            )

            log_run(
                job_name,
                source_name,
                destination_table,
                "WARNING",
                0,
                start_ts,
                information=(
                    f"{message}; "
                    "source file has not arrived or has already "
                    "been archived, so ingestion was skipped."
                )
            )

            return {
                "job": job_name,
                "status": "WARNING",
                "rows": 0,
                "reason": (
                    "SOURCE_FILE_NOT_FOUND"
                ),
            }


        # ===========================================================
        # DETERMINE FILE TYPE
        # ===========================================================

        file_ext = (
            source_file_name
            .rsplit(".", 1)[-1]
            .lower()

            if "." in source_file_name

            else ""
        )


        # ===========================================================
        # EXCEL
        # ===========================================================

        if file_ext in {
            "xlsx",
            "xls"
        }:

            print(
                "FILE FORMAT  : EXCEL"
            )

            df = read_excel_source(
                source_path=source_path,
                source_name=source_name,
                source_file_name=(
                    source_file_name
                ),
            )


            # Unscheduled Trip now lands at:
            # dma.unscheduled_trip
            if (
                source_name.lower()
                == "unscheduled_trip"

                or

                destination_table.lower()
                in {
                    "unscheduled_trip",
                    "bronze_unscheduled_trip",
                }
            ):

                df = (
                    normalize_unscheduled_trip(
                        df
                    )
                )

            schema = df.schema


        # ===========================================================
        # CSV / OTHER FLAT FILE
        # ===========================================================

        else:

            print(
                "FILE FORMAT  : CSV"
            )

            if (
                destination_table
                not in TABLE_SCHEMAS
            ):

                raise ValueError(
                    "No CSV schema "
                    "registered for "
                    "destination_table="
                    f"{destination_table}"
                )

            schema = (
                TABLE_SCHEMAS[
                    destination_table
                ]
            )

            df = (
                spark.read
                .option(
                    "header",
                    True
                )
                .schema(
                    schema
                )
                .csv(
                    source_path
                )
            )


        # ===========================================================
        # COMMON STRING CLEANUP
        # ===========================================================

        for field in (
            schema.fields
        ):

            if isinstance(
                field.dataType,
                StringType
            ):

                c = field.name

                df = (
                    df
                    .withColumn(
                        c,
                        F.when(
                            F.trim(
                                F.col(c)
                            ) == "",
                            None
                        )
                        .otherwise(
                            F.trim(
                                F.col(c)
                            )
                        )
                    )
                )


        # ===========================================================
        # PARSE KNOWN DATE COLUMNS
        # ===========================================================

        for c in (
            DATE_TEXT_COLUMNS
            .get(
                destination_table,
                []
            )
        ):

            if (
                c in df.columns

                and

                dict(
                    df.dtypes
                ).get(c)
                == "string"
            ):

                df = (
                    df
                    .withColumn(
                        c,
                        F.coalesce(
                            F.to_date(
                                F.col(c),
                                "dd-MM-yyyy"
                            ),
                            F.to_date(
                                F.col(c),
                                "MM/dd/yyyy"
                            ),
                            F.to_date(
                                F.col(c),
                                "yyyy-MM-dd"
                            ),
                            F.to_date(
                                F.col(c)
                            ),
                        )
                    )
                )


        # ===========================================================
        # INCREMENTAL FILTER
        # ===========================================================

        if (
            load_type
            == "INCREMENTAL"
        ):

            if not incremental_column:

                raise ValueError(
                    f"{job_name}: "
                    "incremental_column "
                    "is required for "
                    "INCREMENTAL load."
                )


            if (
                incremental_column
                not in df.columns
            ):

                raise ValueError(
                    f"{job_name}: "
                    "incremental column "
                    f"'{incremental_column}' "
                    "not found in source."
                )


            if (
                dict(
                    df.dtypes
                )
                .get(
                    incremental_column
                )
                == "string"
            ):

                df = (
                    df
                    .withColumn(
                        incremental_column,
                        F.coalesce(
                            F.to_date(
                                F.col(
                                    incremental_column
                                ),
                                "dd-MM-yyyy"
                            ),
                            F.to_date(
                                F.col(
                                    incremental_column
                                ),
                                "MM/dd/yyyy"
                            ),
                            F.to_date(
                                F.col(
                                    incremental_column
                                ),
                                "yyyy-MM-dd"
                            ),
                            F.to_date(
                                F.col(
                                    incremental_column
                                )
                            ),
                        )
                    )
                )


            pull_from = gv(
                fields,
                "increment_pull_from_date"
            )

            pull_to = gv(
                fields,
                "increment_pull_to_date"
            )


            if pull_from:

                df = df.filter(
                    F.col(
                        incremental_column
                    )
                    >=
                    F.to_date(
                        F.lit(
                            str(
                                pull_from
                            )
                        )
                    )
                )


            if pull_to:

                df = df.filter(
                    F.col(
                        incremental_column
                    )
                    <=
                    F.to_date(
                        F.lit(
                            str(
                                pull_to
                            )
                        )
                    )
                )


        # ===========================================================
        # AUDIT COLUMNS
        # ===========================================================

        out_df = (
            df
            .withColumn(
                "INSERT_TS",
                F.lit(
                    datetime.now(
                        timezone.utc
                    )
                )
            )
            .withColumn(
                "UPDATE_TS",
                F.lit(None)
                .cast(
                    TimestampType()
                )
            )
            .withColumn(
                "INGESTED_AT",
                F.col(
                    "INSERT_TS"
                )
            )
            .withColumn(
                "UPDATED_AT",
                F.lit(None)
                .cast(
                    TimestampType()
                )
            )
        ).cache()


        row_count = (
            out_df.count()
        )


        num_partitions = (
            sized_partition_count(
                row_count
            )
        )


        table_name = (
            f"{destination_schema}."
            f"{destination_table}"
        )


        # ===========================================================
        # WRITE TARGET
        # ===========================================================

        if (
            load_type
            == "INCREMENTAL"
        ):

            write_mode = "merge"

            merge_incremental_bronze(
                out_df.repartition(
                    num_partitions
                ),
                table_name,
                key_columns,
            )

        else:

            write_mode = "overwrite"

            writer = (
                out_df
                .repartition(
                    num_partitions
                )
                .write
                .format(
                    "delta"
                )
                .mode(
                    write_mode
                )
                .option(
                    "overwriteSchema",
                    "true"
                )
            )

            write_table_with_retry(
                writer,
                table_name,
            )


        out_df.unpersist()

        out_df = None


        print(
            f"DATA WRITE    : SUCCESS "
            f"({row_count} rows)"
        )


        # ===========================================================
        # ARCHIVE + DELETE SOURCE
        #
        # This happens ONLY after Delta write succeeds.
        # ===========================================================

        archive_path = (
            archive_and_delete_source(
                source_container=(
                    source_container
                ),
                source_folder=(
                    source_folder
                ),
                source_file_name=(
                    source_file_name
                ),
                archive_container=(
                    archive_container
                ),
                archive_folder=(
                    archive_folder
                ),
            )
        )


        # ===========================================================
        # SUCCESS LOG
        # ===========================================================

        log_run(
            job_name,
            source_name,
            destination_table,
            "SUCCESS",
            row_count,
            start_ts,
            information=(
                f"Loaded {row_count} rows; "
                f"source={source_path}; "
                f"destination={table_name}; "
                f"mode={write_mode}; "
                f"keys={key_columns}; "
                f"partitions={num_partitions}; "
                f"file_format={file_ext}; "
                f"archive={archive_path}; "
                "source_deleted=true"
            )
        )


        print(
            f"SUCCESS: "
            f"{job_name} "
            f"({row_count} rows, "
            f"{num_partitions} "
            "partition(s))"
        )


        return {
            "job": job_name,
            "status": "SUCCESS",
            "rows": row_count,
            "archive_path": (
                archive_path
            ),
        }


    except Exception as exc:

        if out_df is not None:

            try:

                out_df.unpersist()

            except Exception:

                pass


        error_text = (
            f"{exc}\n"
            f"{traceback.format_exc()}"
        )


        log_run(
            (
                job_name
                or source_name
                or "unknown_file_job"
            ),
            (
                source_name
                or "unknown"
            ),
            (
                destination_table
                or "unknown"
            ),
            "FAILED",
            0,
            start_ts,
            error_message=(
                error_text[:8000]
            )
        )


        print(
            f"FAILED: "
            f"{job_name or source_name} "
            f"- {exc}"
        )


        return {
            "job": (
                job_name
                or source_name
                or "unknown_file_job"
            ),
            "status": "FAILED",
            "rows": 0,
            "error": str(exc),
        }


In [ ]:
# -------------------------------------------------------------------
# Main driver
# -------------------------------------------------------------------
p_source_key_clean = (p_source_key or "").strip()
p_source_key_escaped = p_source_key_clean.replace("'", "''")

source_filter_clause = ""
if p_source_key_clean:
    source_filter_clause = f"""
      AND (
            LOWER(SOURCE_TABLE_NAME) = LOWER('{p_source_key_escaped}')
         OR LOWER(DESTINATION_TABLE_NAME) = LOWER('{p_source_key_escaped}')
         OR CAST(ROW_ID AS STRING) = '{p_source_key_escaped}'
      )
    """

# Alias the new control-table columns to the names already expected by the
# existing ingestion functions. This lets us migrate metadata first without
# rewriting the working Bronze ingestion logic.
control_df = spark.sql(f"""
SELECT
    ROW_ID,
    TRANSFER_TYPE,
    TRANSFER_TYPE AS ingestion_method,
    SOURCE_SYSTEM,
    CASE
        WHEN TRANSFER_TYPE = 'API_TO_BRZ' THEN 'API'
        ELSE 'FLAT_FILE'
    END AS source_type,
    SOURCE_SCHEMA,
    SOURCE_TABLE_NAME,
    SOURCE_TABLE_NAME AS source_name,
    SOURCE_FILE_NAME,
    PARAM_1 AS source_container,
    PARAM_2 AS source_folder,
    ARCHIVE_CONTAINER AS archive_container,
    ARCHIVE_FOLDER AS archive_folder,
    DESTINATION_SCHEMA,
    DESTINATION_TABLE_NAME,
    DESTINATION_TABLE_NAME AS destination_table,
    LOAD_TYPE,
    PRIMARY_KEY,
    INCREMENTAL_COLUMN,
    EXECUTION_ORDER,
    JOB_DEPENDENCIES,
    RUN_ACTIVE_FLAG,
    DESTINATION_TABLE_NAME AS job_name,
    PARAM_1,
    PARAM_2,
    INSERT_TS,
    UPDATE_TS
FROM {CONTROL_TABLE}
WHERE RUN_ACTIVE_FLAG = 1
  AND TRANSFER_TYPE IN ('FILE_TO_BRZ', 'API_TO_BRZ')
  {source_filter_clause}
ORDER BY EXECUTION_ORDER
""")

control_rows = control_df.collect()

print(f"Execution ID: {RUN_ID}")
print(f"Active Bronze jobs found: {len(control_rows)}")

if not control_rows:
    raise ValueError(
        f"No active Bronze rows found in {CONTROL_TABLE}."
    )

results = []

for row in control_rows:
    row_fields = row_dict(row)

    if is_api_row(row_fields):
        result = ingest_vehicle_api_row(row)
    else:
        result = ingest_csv_row(row)

    results.append(result)

successes = [r for r in results if r["status"] == "SUCCESS"]
warnings = [r for r in results if r["status"] == "WARNING"]
failures = [r for r in results if r["status"] == "FAILED"]


print(
    f"Bronze run complete. "
    f"Success={len(successes)} "
    f"Warning={len(warnings)} "
    f"Failed={len(failures)}"
)

for result in results:
    print(result)

if failures:
    failed_jobs = ", ".join(r["job"] for r in failures)
    raise RuntimeError(
        f"Bronze ingestion failed for: {failed_jobs}"
    )